# Getting Started with Sounio: Knowledge & GUM Arithmetic

This notebook introduces the core concepts of epistemic computing in Sounio:
- Creating **Knowledge** objects with uncertainty
- **GUM arithmetic**: propagating uncertainty through calculations
- Running Sounio code from Python
- Parsing epistemic results

## Part 1: Creating Knowledge Objects

A `Knowledge` object represents a measured value with standard uncertainty (GUM k=1).

In [ ]:
import sys
sys.path.insert(0, '../sounio-py/python')

from sounio.knowledge import Knowledge

# Create a simple Knowledge value: 500.0 mg ± 2.5 mg
dose = Knowledge(500.0, 2.5, "calibration_batch_2026")
print(f"Dose: {dose}")
print(f"Value: {dose.value}")
print(f"Uncertainty: {dose.epsilon}")
print(f"Provenance: {dose.provenance}")
print(f"Relative uncertainty: {dose.relative_uncertainty:.2%}")

In [ ]:
# Create multiple measurements
half_life = Knowledge(4.62, 0.767, "pk_fit_study_001")
clearance = Knowledge(12.5, 1.5, "pk_fit_study_001")
bioavailability = Knowledge(0.85, 0.05, "in_vivo_data")

print("Pharmacokinetic Parameters:")
print(f"  Half-life: {half_life}")
print(f"  Clearance: {clearance}")
print(f"  Bioavailability: {bioavailability}")
print()
print("Relative uncertainties:")
print(f"  Half-life: {half_life.relative_uncertainty:.2%}")
print(f"  Clearance: {clearance.relative_uncertainty:.2%}")
print(f"  Bioavailability: {bioavailability.relative_uncertainty:.2%}")

In [ ]:
# Check reliability (default threshold: 5%)
print("Reliability assessment (5% threshold):")
for name, k in [("Half-life", half_life), ("Clearance", clearance), ("Bioavailability", bioavailability)]:
    is_reliable = k.is_reliable(threshold=0.05)
    print(f"  {name}: {'✓ Reliable' if is_reliable else '✗ Not reliable'}")

## Part 2: GUM Uncertainty Propagation

The **GUM (Guide to the Expression of Uncertainty in Measurement)** standard defines how uncertainty propagates through arithmetic operations.

In [ ]:
# Addition / Subtraction: ε = sqrt(εa² + εb²)
dose_a = Knowledge(200.0, 5.0, "batch_A")
dose_b = Knowledge(300.0, 8.0, "batch_B")
total_dose = dose_a + dose_b

print("Addition (GUM: ε_total = sqrt(εa² + εb²)):")
print(f"  {dose_a} +")
print(f"  {dose_b} =")
print(f"  {total_dose}")
print(f"  Expected ε: sqrt(5² + 8²) = {(5**2 + 8**2)**0.5:.2f}")

In [ ]:
# Multiplication: ε = |a·b| · sqrt((εa/a)² + (εb/b)²)
# Example: dose × body weight
dose_mg = Knowledge(500.0, 10.0, "scale_calibration")  # 500 mg ± 10 mg (2%)
weight_kg = Knowledge(70.0, 2.0, "scale_calibration")  # 70 kg ± 2 kg (2.86%)
dose_per_kg = dose_mg / weight_kg

print("Multiplication / Division (GUM: relative uncertainties combine):")
print(f"  Dose: {dose_mg}")
print(f"  Weight: {weight_kg}")
print(f"  Dose/kg: {dose_per_kg}")
print()
print(f"  Relative uncertainty (Dose): {dose_mg.relative_uncertainty:.2%}")
print(f"  Relative uncertainty (Weight): {weight_kg.relative_uncertainty:.2%}")
print(f"  Relative uncertainty (Dose/kg): {dose_per_kg.relative_uncertainty:.2%}")

In [ ]:
# Scalar multiplication: ε = |factor| · ε
blood_sample_1 = Knowledge(100.0, 5.0, "measurement")
diluted_3x = blood_sample_1 * 3

print("Scalar multiplication (dilution):")
print(f"  Original: {blood_sample_1}")
print(f"  Diluted 3×: {diluted_3x}")
print()
print(f"  Uncertainty scales linearly: 5 × 3 = {5 * 3}")

In [ ]:
# Chain of calculations (provenance tracking)
conc_a = Knowledge(10.0, 0.5, "measurement_A")
conc_b = Knowledge(20.0, 1.0, "measurement_B")
conc_c = Knowledge(15.0, 0.7, "measurement_C")

mean_conc = (conc_a + conc_b + conc_c) / 3

print("Chained calculations (provenance chain):")
print(f"  Concentration A: {conc_a}")
print(f"  Concentration B: {conc_b}")
print(f"  Concentration C: {conc_c}")
print()
print(f"  Mean: {mean_conc}")
print(f"  Provenance: {mean_conc.provenance}")

## Part 3: Serialization & Sounio Format

Knowledge objects can be serialized to the canonical Sounio format for round-tripping with the compiler.

In [ ]:
# Serialize to Sounio canonical format
k = Knowledge(500.0, 2.5, "calibration_batch_2026")
sounio_format = k.to_sounio_format()
print("Sounio canonical format:")
print(f"  {sounio_format}")

In [ ]:
# Parse from Sounio output
sounio_output = 'Knowledge { value: 500 epsilon: 2.5 prov: "calibration_batch_2026" }'
parsed = Knowledge.from_sounio_output(sounio_output)
print("Parsed from Sounio output:")
print(f"  {parsed}")
print()
print("Verify round-trip:")
print(f"  Original: {k}")
print(f"  Parsed:   {parsed}")
print(f"  Match:    {k == parsed}")

In [ ]:
# Dictionary serialization (for JSON, databases, etc.)
k_dict = k.to_dict()
print("Dictionary representation:")
print(f"  {k_dict}")
print()

k_restored = Knowledge.from_dict(k_dict)
print(f"Restored: {k_restored}")
print(f"Match: {k == k_restored}")

## Summary

In this notebook, you learned:
1. **Knowledge objects** encapsulate measured values with uncertainty and provenance
2. **GUM arithmetic** automatically propagates uncertainty through calculations
3. **Relative uncertainty** quantifies measurement quality (lower is better)
4. **Serialization** enables round-tripping with Sounio programs and JSON/database storage

Next: See notebook 02 for visualization and plotting.